# OsteoLab — 1. Preparación de datos

Este es el primer cuaderno de tres. Si lo estás viendo tal cual llega a la charla de PyCon: no
hace falta saber nada de machine learning para seguirlo, solo te pedimos paciencia — vamos a ir
**muy despacio**, paso a paso, explicando cada decisión antes de mostrar el código que la
implementa.

**La pregunta que responde este cuaderno es simple**: si quiero que un programa reconozca huesos
en una foto, ¿de dónde saco las imágenes para enseñarle, y cómo las convierto en números que un
modelo pueda usar?

Los otros dos cuadernos de la serie:

- `pycon26_02_entrenamiento.ipynb` — cómo se entrenan los modelos con estos datos.
- `pycon26_03_inferencia.ipynb` — cómo se usan esos modelos ya entrenados para clasificar una foto
  nueva, y cómo se expone todo esto como una API.

**Un principio que vamos a repetir varias veces**: este cuaderno no reimplementa nada por su
cuenta. Llama exactamente al mismo código que usa la aplicación en producción
(`shared/`, `scripts/`). Lo que ves ejecutarse aquí es, literalmente, lo que ejecuta el sistema
real — no es una simplificación "de charla" que luego funcione distinto.

## 0. El problema, en una frase

Queremos clasificar huesos a partir de una foto: dado una imagen, decir a qué **grupo
morfológico** pertenece (cráneo, fémur-tipo-hueso-largo, vértebra, costilla...).

Para que un modelo aprenda esto necesita ejemplos — muchos, y variados. Y aquí está el primer
obstáculo real del proyecto: **conseguir miles de fotos de huesos reales, bien etiquetadas, es
carísimo**. No hay un dataset público decente para esto. La solución de este proyecto es generar
la mayoría de los datos de entrenamiento de forma **sintética**, a partir de modelos 3D, y usar
solo un número pequeño de fotos reales para "anclar" el modelo al mundo real.

Ese es el hilo de todo este cuaderno: de dónde salen las imágenes, por qué hay varias "versiones"
de las mismas imágenes, y cómo se convierten en una tabla de números que un modelo puede
consumir.

## 1. Arrancamos: ubicar el proyecto

Antes de nada, el cuaderno necesita saber dónde está la raíz del repositorio, para poder importar
el código del proyecto igual que lo haría la aplicación real. Esta celda sube directorios hasta
encontrar `params.yaml`, que es un archivo que solo existe en la raíz.

In [ ]:
# Nada de machine learning todavía: solo ubicarnos en el disco.
import os
import sys
from pathlib import Path


def find_repo_root():
    # Busca hacia arriba desde donde esté corriendo el notebook hasta encontrar
    # `params.yaml`, que marca la raíz del repositorio.
    start = Path.cwd()
    for candidate in (start, *start.parents):
        if (candidate / "params.yaml").exists() and (candidate / "training").is_dir():
            return candidate
    raise RuntimeError("No encuentro la raíz del repo. Abre el notebook desde dentro de él.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Raíz del proyecto:", REPO_ROOT)

In [ ]:
# Tres carpetas que vamos a usar todo el rato: les damos un nombre corto.
PROCESSED = REPO_ROOT / "data/processed"       # tablas (CSV) listas para entrenar
BONE_DB = REPO_ROOT / "base_datos_osea"        # imágenes: renders sintéticos + fotos reales
MODELS = REPO_ROOT / "models"                  # modelos ya entrenados (los usaremos en el cuaderno 2 y 3)

In [ ]:
import subprocess


def run(cmd, enabled, cwd=REPO_ROOT):
    # Lanza un script real del proyecto y muestra su salida.
    # `enabled=False` por defecto: construir los datasets desde cero tarda minutos u horas
    # (hay que renderizar y medir miles de imágenes), así que en la charla dejamos las celdas
    # "apagadas" y solo mostramos qué comando lanzarían. Si quieres reproducirlo en tu máquina,
    # cambia el `False` a `True` en la celda de interruptores de abajo.
    printable = " ".join(str(c) for c in cmd)
    if not enabled:
        print(f"[celda desactivada, el comando sería]\n$ {printable}")
        return None
    print(f"$ {printable}\n")
    proc = subprocess.run([sys.executable, *cmd[1:]] if cmd[0] == "python" else cmd,
                          cwd=cwd, text=True, capture_output=True)
    print(proc.stdout[-8000:])
    if proc.returncode != 0:
        print("STDERR:\n", proc.stderr[-4000:])
    return proc


# Único interruptor caro de este cuaderno: reconstruir las tablas de features desde cero.
RUN_BUILD_DATASETS = False

## 2. De dónde salen las imágenes

No hay una carpeta `data/raw/<clase_de_hueso>/foto1.jpg` con miles de fotos reales — eso no
existe para este problema. Lo que hay es:

1. Un **catálogo de mallas 3D** de huesos reales, verificadas a mano una por una, descargadas de
   repositorios científicos (MorphoSource, Sketchfab, Smithsonian). Cada malla es la forma 3D de
   un espécimen concreto: *este* fémur de *este* individuo.
2. Un **motor de render** (Blender) que coloca una cámara alrededor de cada malla y toma 24 fotos
   desde ángulos distintos. Así, de una sola malla 3D sacamos 24 imágenes de entrenamiento.
3. Por encima de eso, varias **recetas de render** cada vez más realistas (lo vemos en el punto
   siguiente), porque un render demasiado "limpio" no se parece lo bastante a una foto real.
4. Un conjunto pequeño, curado a mano, de **fotos reales** de huesos — pocas, pero son las que de
   verdad importan para medir si el modelo funciona fuera del laboratorio.

La taxonomía que se aprende tiene **9 grupos morfológicos**: `cranio`, `mandibula_maxilar`,
`hueso_largo`, `hueso_plano`, `pelvis`, `sacro`, `vertebra`, `costilla`, `hueso_pequeno`. No son
especies ni huesos exactos — son *formas*. Con solo 1 a 9 especímenes físicos por hueso, pedirle
al modelo que distinga 21 huesos exactos (fémur vs. húmero vs...) haría que aprendiera a reconocer
*ese objeto concreto de la foto*, no la forma en general. Esto se comprobó de verdad: dejando
fuera especímenes completos durante la validación (en vez de solo fotos), la métrica honesta caía
mucho frente a la optimista.

In [ ]:
# ¿Cuántas imágenes tenemos en cada receta de render?
for carpeta in sorted(BONE_DB.glob("renders*")):
    n_imagenes = sum(1 for _ in carpeta.rglob("*.png")) + sum(1 for _ in carpeta.rglob("*.jpg"))
    print(f"{carpeta.name:32s} {n_imagenes:6d} imágenes")

In [ ]:
# Y cuántas fotos reales curadas hay por grupo (las que de verdad importan).
curated = BONE_DB / "Fotos entrenamiento" / "_curated"
if curated.exists():
    for grupo in sorted(p for p in curated.iterdir() if p.is_dir()):
        n = len(list(grupo.glob("*")))
        print(f"{grupo.name:18s} {n:4d} fotos reales")
else:
    print("Esta copia del repo no tiene las fotos reales curadas descargadas.")

### Por qué hay varias "recetas" de render (y un experimento que NO funcionó)

Esto es el núcleo del problema **sim2real** (de simulación a mundo real): un modelo entrenado solo
con siluetas perfectas sobre fondo blanco aprende a reconocer *siluetas perfectas sobre fondo
blanco*, no huesos. En cuanto le enseñas una foto real — con sombras, polvo, una mesa de madera
detrás — falla. Por eso se fueron probando recetas de render cada vez más realistas:

| Receta | Carpeta | Qué añadía |
|---|---|---|
| `renders` | `renders/` | el punto de partida: silueta limpia, fondo plano |
| `renders_noisy_composited` | `renders_noisy_composited/` | el hueso recortado y pegado sobre fondos reales fotografiados |
| `renders_noisy_bproc` | `renders_noisy_bproc/` | BlenderProc: iluminación física, sombras, material con textura |
| `renders_lab` | `renders_lab/` | escena de laboratorio completa: física real (el hueso cae y se asienta sobre una mesa), HDRI, y el material multicapa más completo (`bone_material.py`) |

El dataset de producción (`combined_lab`) mezcla `renders_lab` + `noisy_composited` +
`noisy_bproc` + fotos reales. Al revisarlo para esta charla se notó algo llamativo: casi todos los
especímenes de `noisy_composited` y `noisy_bproc` (112/113 y 90/91) son los **mismos huesos
físicos** que ya están en `renders_lab`, solo con una piel sintética distinta — en teoría eso
sobrepondera esos especímenes concretos en vez de ampliar la variedad.

**Se probó la "corrección" obvia — quitar las dos variantes redundantes y quedarse solo con
`renders_lab` + reales — y empeoró, no mejoró**: accuracy en el banco grande de fotos reales
(117 fotos) bajó de 0.8120 a 0.6923 (-12 puntos). La hipótesis de la duplicación era razonable,
pero en la práctica pesó más la pérdida de volumen y variedad visual (el dataset pasó de 2938 a
1447 filas) que el sesgo de repetir especímenes. **Se revirtió el cambio** — el dataset de
producción sigue mezclando las tres variantes sintéticas. Queda documentado como ejemplo de que
una intuición metodológicamente correcta no siempre gana en la práctica: hay que medir, no basta
con razonar.

### Criterio de curación de las fotos reales

Con tan pocas fotos reales disponibles, el criterio de qué se acepta importa mucho. La regla es:
**se filtra por grupo morfológico, no por especie**. Una mandíbula de perro cuenta igual que una
mandíbula humana, porque lo que el modelo aprende es la *forma* de "mandíbula", no de qué animal
es. Lo que sí descalifica una foto:

- el hueso no se distingue bien del fondo (demasiada sombra, mal enfoque),
- está roto o fragmentado de forma que ya no representa la forma típica del grupo,
- está mal etiquetado (alguien marcó "vértebra" algo que es otra cosa).

El registro de qué se acepta y por qué vive en `data/real_photos_accepted/manifest.csv`, y se
revisa con una pequeña app interna (`scripts/review_app.py`) en vez de a ojo sobre carpetas.

## 3. De una imagen a números: features geométricas con OpenCV

Un modelo no "ve" una foto como nosotros. Necesita números. El paso que convierte una imagen en
números se llama **extracción de features** y, en este proyecto, es puramente geometría clásica
— nada de aprendizaje automático todavía. Es el mismo paso, exactamente el mismo código, que usa
la API en producción cuando alguien sube una foto.

La función que hace esto es `extract_features`, y vive en un único sitio del proyecto
(`shared/cv_extractor/extract.py`) para que entrenamiento e inferencia midan **siempre igual**.
Vamos a verla en acción, paso a paso, sobre una imagen de ejemplo.

In [ ]:
# Elegimos una imagen de ejemplo para seguir el resto del cuaderno sobre ella.
sample = next(iter(sorted((BONE_DB / "renders_lab").rglob("*.png"))), None)
if sample is None:
    sample = next(iter(sorted(curated.rglob("*.jpg"))))

print("Imagen de ejemplo:", sample.relative_to(REPO_ROOT))

In [ ]:
import pandas as pd

from shared.cv_extractor.extract import extract_features

# backend="classic" = segmentación clásica (sin aprendizaje), la que usamos en renders limpios.
with open(sample, "rb") as fh:
    feats = extract_features(fh, backend="classic")

# El diccionario trae también imágenes codificadas en base64 y el recorte — las dejamos para
# las celdas siguientes, aquí solo queremos las medidas.
solo_numeros = {k: v for k, v in feats.items() if not k.endswith("_base64") and k != "crop_image"}

# pd.Series convierte el dict en una tabla de una columna (feature -> valor), más legible que un print.
pd.Series(solo_numeros, name="valor").rename_axis("feature").to_frame()

Antes de seguir, merece la pena parar un momento en qué significa cada grupo de esos números:

- **Geometría base**: tamaño del hueso en la foto (`area_px`), proporción respecto al cuadro que
  lo encierra (`aspect_ratio`, `extent`), y si la silueta está "llena" o tiene huecos
  (`solidity`).
- **Forma**: `elongation` — qué tan alargado es, calculado con PCA sobre el contorno. Es, como
  veremos en la sección 5, la feature que más importa de todas. También `circularity`,
  `convexity`, y los 7 `hu_moments` (una firma matemática clásica de reconocimiento de formas,
  invariante a rotación y escala).
- **Perfil de anchura**: 20 cortes perpendiculares al eje largo del hueso, para capturar si se
  ensancha en los extremos (como un fémur) o es uniforme (como una costilla).
- **Curvatura**: cuánto se curva la línea central del hueso.
- **Visual**: versiones de la imagen con las medidas dibujadas encima, para poder inspeccionar
  que la segmentación fue correcta — eso es justo lo que mostramos ahora.

Un detalle importante: todas estas medidas son relativas a la imagen (píxeles, proporciones). No
hay forma de saber el tamaño real en centímetros sin una referencia de escala en la foto, así que
el proyecto ni lo intenta.

In [ ]:
from IPython.display import HTML, display

# Dos imágenes: la original con las medidas dibujadas encima, y la máscara que detectó el contorno.
display(HTML("".join(
    f'<img src="data:image/png;base64,{feats[k]}" style="max-width:45%;margin:4px">'
    for k in ("annotated_image_base64", "mask_overlay_base64") if feats.get(k)
)))

In [ ]:
# El recorte final: exactamente lo que ve la CNN más adelante (fondo gris fuera del hueso).
import numpy as np
from PIL import Image

with open(sample, "rb") as fh:
    con_recorte = extract_features(fh, backend="classic", include_crop=True)

crop = con_recorte.get("crop_image")
if crop is not None:
    print("Forma del recorte:", getattr(crop, "shape", type(crop)))
    display(Image.fromarray(np.asarray(crop)[:, :, ::-1]))

### Tres formas de recortar el hueso del fondo

El paso de "encontrar dónde está el hueso en la imagen" (segmentación) se puede hacer de tres
maneras, y la elección afecta mucho más a las fotos reales que a los renders:

| Backend | Cómo funciona | Cuándo se usa |
|---|---|---|
| `classic` | Umbralización de Otsu + contornos — geometría pura, sin aprendizaje | renders con fondo limpio y uniforme |
| `fastsam` | Red FastSAM pre-entrenada | alternativa rápida sobre fotos reales |
| `grounded_sam` | Se le da una descripción de texto ("a bone") y SAM segmenta a partir de ahí | **el que se usa por defecto en fotos reales**, el más robusto a sombras y texturas |

Una segmentación mala no solo da un recorte feo: rompe features que dependen de un contorno fino
(contar agujeros, medir circularidad con precisión). Por eso, como veremos en el cuaderno de
inferencia, la regla rápida que se usa en producción solo confía en `elongation` — es la medida
que mejor resiste a una segmentación imperfecta.

## 4. De imágenes sueltas a una tabla de entrenamiento

Ya sabemos medir *una* imagen. El siguiente paso es repetir esto sobre miles de imágenes y volcar
el resultado en una tabla (un CSV): una fila por imagen, una columna por medida, más la etiqueta
(a qué grupo pertenece ese hueso), deducida de la ruta de la carpeta
(`renders/<especie>/<hueso>/<espécimen>/vista_03.png`).

Hay un **script constructor distinto por cada receta de render** (de la sección 2), y uno más que
combina varias recetas en una sola tabla.

In [ ]:
# Cada línea construye (o reconstruiría, si RUN_BUILD_DATASETS=True) una tabla distinta.
run(["python", "-m", "shared.cv_extractor.build_dataset"], RUN_BUILD_DATASETS)             # renders/
run(["python", "-m", "shared.cv_extractor.build_dataset_lab"], RUN_BUILD_DATASETS)         # renders_lab/
run(["python", "-m", "shared.cv_extractor.build_dataset_bproc"], RUN_BUILD_DATASETS)       # renders_noisy_bproc/
run(["python", "-m", "shared.cv_extractor.build_dataset_real_photos", "--split", "train"], RUN_BUILD_DATASETS)
run(["python", "-m", "shared.cv_extractor.build_dataset_real_photos", "--split", "test"], RUN_BUILD_DATASETS)
run(["python", "scripts/build_combined_lab_variant.py"], RUN_BUILD_DATASETS)               # mezcla las 3 variantes + reales (ver nota de arriba: el intento de quitar duplicados empeoró el resultado)

### La trampa que hay que evitar: fuga de datos entre train y test

Cada malla 3D produce 24 vistas. Si repartimos esas 24 vistas al azar entre entrenamiento y test,
el modelo acaba viendo el *mismo objeto* en ambos lados, solo desde ángulos distintos — y la
métrica de evaluación sale artificialmente alta, porque no está midiendo si generaliza, sino si
reconoce ese espécimen concreto.

La solución: un conjunto de especímenes completos (`BENCHMARK_SPECIMENS`, dentro de
`build_dataset.py`) se reserva entero para el banco de evaluación, nunca entra en la tabla de
entrenamiento. Es la misma lógica de fondo que justificó, en la sección 2, usar grupos
morfológicos y no huesos exactos: **evitar que el modelo memorice objetos concretos en vez de
aprender formas**.

In [ ]:
import pandas as pd

# Tamaño de cada tabla que ya existe en el repo.
for nombre in ["bone_geometric_features.csv", "bone_geometric_features_lab.csv",
               "bone_geometric_features_combined.csv", "bone_geometric_features_combined_lab.csv",
               "bone_geometric_benchmark.csv", "bone_geometric_features_real_photos_test.csv"]:
    ruta = PROCESSED / nombre
    if ruta.exists():
        df = pd.read_csv(ruta)
        print(f"{nombre:48s} {len(df):6d} filas  {len(df.columns):3d} columnas")

La tabla que de verdad alimenta el modelo en producción es la **combinada**:
`bone_geometric_features_combined_lab.csv`, que mezcla `renders_lab` + las dos variantes
"ruidosas" + las fotos reales de entrenamiento (2938 filas). Esta receta se llama `hybrid_lab` —
y, pese a la duplicación de especímenes descrita arriba, es la que mejor funciona de las probadas
hasta ahora.

## 5. Explorar la tabla antes de entrenar nada

Con la tabla ya construida, antes de entrenar cualquier modelo conviene mirarla: ¿están las clases
balanceadas?, ¿hay suficientes especímenes distintos por clase?, ¿alguna medida separa bien las
clases a simple vista?

In [ ]:
dataset_path = PROCESSED / "bone_geometric_features_combined_lab.csv"
if not dataset_path.exists():
    dataset_path = PROCESSED / "bone_geometric_features.csv"

df = pd.read_csv(dataset_path)
print("Tabla usada:", dataset_path.name, "—", df.shape, "(filas, columnas)")
df.groupby("bone_group").size().sort_values(ascending=False)

In [ ]:
# ¿Cuántos especímenes físicos distintos hay detrás de cada grupo? (no solo cuántas fotos)
if "specimen" in df.columns:
    print(df.groupby("bone_group")["specimen"].nunique().to_string())

In [ ]:
import matplotlib.pyplot as plt

# Histograma de las features más interesantes, una clase encima de otra, a ver cuáles separan bien.
interesantes = [c for c in ["elongation", "aspect_ratio", "circularity", "solidity", "convexity",
                             "curvature_mean", "mean_width", "std_width"] if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, col in zip(axes.ravel(), interesantes):
    for grupo, sub in df.groupby("bone_group"):
        ax.hist(sub[col].dropna(), bins=25, alpha=0.4, label=grupo)
    ax.set_title(col, fontsize=9)
axes.ravel()[0].legend(fontsize=6)
plt.tight_layout()
plt.show()

Fijaos en el histograma de `elongation`: es la única medida con una frontera limpia entre
`hueso_largo` y el resto, y además — lo comprobaremos en el cuaderno de inferencia — es la única
que **sigue separando bien en fotos reales**, no solo en renders sintéticos limpios. El resto de
medidas separan de maravilla en el banco sintético y se degradan mucho en foto real. Esa
observación, por sí sola, es la que justifica la regla rápida ("cascada") que veremos después: si
algo mide tan limpio como `elongation`, no hace falta gastar un modelo entero en decidirlo.

In [ ]:
# El mismo espacio PCA de 2 dimensiones que la API expone en GET /pca/reference,
# para poder visualizar las ~9 clases en un solo plano.
ref_path = PROCESSED / "bone_geometric_pca_reference.csv"
if ref_path.exists():
    ref = pd.read_csv(ref_path)
    fig, ax = plt.subplots(figsize=(7, 6))
    for grupo, sub in ref.groupby("bone_group"):
        ax.scatter(sub["pca_x"], sub["pca_y"], s=8, alpha=0.6, label=grupo)
    ax.legend(fontsize=7)
    ax.set_title("Proyección PCA (2 componentes) de las features estandarizadas")
    plt.show()
else:
    print("El PCA de referencia se genera al entrenar el modelo geométrico (cuaderno 2).")

### ¿Hacen falta todas estas features?

Cuidado con una confusión frecuente: el PCA de arriba explica **varianza**, no **utilidad para
clasificar**. Una feature puede variar mucho entre filas y no servir para nada (ruido), y otra
puede tener poca varianza y ser justo la que separa las clases (como vimos con `elongation`). Para
responder "¿esta columna hace falta?" hay dos herramientas más útiles que el PCA:

1. **Correlación entre features** — si dos columnas van casi siempre pegadas, una es redundante:
   no aporta información nueva, solo ruido y coste de cómputo.
2. **`feature_importances_` del Random Forest** (sección 7 del cuaderno 2) o, mejor,
   **permutation importance** — miden impacto real en la predicción, no solo varianza.

El PCA sigue siendo útil para otra cosa: ver qué features *dominan* cada componente (los
`loadings`), que es una forma rápida de detectar grupos de columnas que miden básicamente lo mismo.


In [ ]:
import numpy as np

# Matriz de correlación entre las features numéricas: valores cerca de 1 o -1 = redundancia.
cols_numericas = [c for c in df.select_dtypes("number").columns if c not in {"area_px"}]
corr = df[cols_numericas].corr()

# Nos quedamos solo con pares muy correlacionados (umbral 0.9), sin repetir cada par dos veces.
pares_redundantes = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .loc[lambda s: s.abs() > 0.9]
        .sort_values(ascending=False)
)
print("Pares de features casi redundantes (|correlación| > 0.9):")
print(pares_redundantes.to_string() if not pares_redundantes.empty else "ninguno por encima del umbral")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Mismo PCA de 2 componentes que la API, pero calculado aquí para poder leer los "loadings":
# cuánto pesa cada feature original en cada componente.
X = df[cols_numericas].dropna()
X_std = StandardScaler().fit_transform(X)
pca = PCA(n_components=2).fit(X_std)

loadings = pd.DataFrame(pca.components_.T, index=cols_numericas, columns=["PC1", "PC2"])
print("Varianza explicada por componente:", pca.explained_variance_ratio_.round(3))
loadings.sort_values("PC1", key=abs, ascending=False).head(10)

### Mini-ablación: ¿sirven de algo las features geométricas, por sí solas?

Antes de entrenar ningún modelo "de verdad" (eso es el cuaderno 2), una comprobación rápida y
barata: si entrenamos un clasificador simple **solo** con las columnas de `extract_features` —
sin ver la imagen en ningún momento — ¿le ganamos a un baseline que no mira nada y siempre
responde la clase más frecuente?

Si la respuesta fuera "no", todo el trabajo de medir geometría (sección 3) sería inútil. Lo
comprobamos con la misma disciplina que ya vimos antes: evaluamos con especímenes completos
fuera del entrenamiento (`GroupKFold`), no con un split aleatorio por fila — si no, la pregunta
estaría mal planteada desde el principio.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_score

datos_ablacion = df.dropna(subset=cols_numericas + ["bone_group"])
X_abl = datos_ablacion[cols_numericas]
y_abl = datos_ablacion["bone_group"]

# Si no hay columna "specimen", agrupamos por bone+species (la unidad real que no debe fugarse).
if "specimen" in datos_ablacion.columns:
    grupos = datos_ablacion["specimen"]
else:
    grupos = datos_ablacion.get("bone", y_abl)

cv = GroupKFold(n_splits=5)

baseline = DummyClassifier(strategy="most_frequent")
geometrico = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)

acc_baseline = cross_val_score(baseline, X_abl, y_abl, cv=cv, groups=grupos, scoring="accuracy")
acc_geometrico = cross_val_score(geometrico, X_abl, y_abl, cv=cv, groups=grupos, scoring="accuracy")

pd.DataFrame({
    "baseline (clase más frecuente)": [acc_baseline.mean()],
    "solo geometría (Random Forest)": [acc_geometrico.mean()],
}, index=["accuracy media (5-fold, especímenes fuera)"])

Con 9 clases, el baseline tonto ronda el 10-15% (depende de qué tan desbalanceado esté el
dataset, ver sección 5). Si el Random Forest solo-geometría se queda muy por encima de eso, la
respuesta a la pregunta de arriba es que sí: la geometría por sí sola ya es una señal fuerte,
antes de que una CNN mire un solo píxel.

Esto no es todavía la pregunta completa: lo que de verdad importa en producción es si el vector
geométrico **añade algo a la CNN**, no si funciona solo. Esa comparación — `cnn_only` vs.
`hybrid`, con y sin el vector geométrico — es más cara (necesita entrenar la red) y la hacemos
con todo detalle en el cuaderno 2, sección "Ablación: ¿aporta algo el vector geométrico?".


## 6. Un primer filtro, antes de clasificar nada: ¿esto es un hueso?

Antes incluso de medir geometría, el sistema real se hace una pregunta más básica cuando alguien
sube una foto: **¿esto es un hueso, siquiera?** Este filtro no usa ninguno de los modelos que
entrenaremos — usa CLIP, un modelo ya entrenado por terceros que entiende imágenes y texto a la
vez, en modo *zero-shot* (sin entrenamiento propio para esta tarea concreta).

La idea: se comparan los "embeddings" (representación numérica interna) de la foto subida contra
los de un puñado de frases fijas como "a photo of a bone" frente a "a photo of something that is
not a bone". Si gana claramente la opción de "no es un hueso", el flujo se detiene ahí mismo — no
se gasta ni segmentación ni ningún modelo más caro.

In [ ]:
from shared.clip_filter.predict import classify_image as clip_classify

with open(sample, "rb") as fh:
    print(clip_classify(fh))

Existe también un segundo filtro, entrenado aparte con datos propios: una ResNet18 binaria
(hueso / no-hueso), expuesta todavía como endpoint suelto (`POST /filter/is-bone`) pero que **ya no
forma parte del flujo por defecto** — CLIP cubre la misma función sin necesitar un dataset propio
de "cosas que no son huesos". Se deja aquí solo como referencia histórica; no se usa en producción.

In [ ]:
# Esta celda está desactivada a propósito: documenta cómo se entrenó ese filtro, no se re-entrena aquí.
run(["python", "shared/bone_filter/train.py", "--epochs", "15", "--batch-size", "64", "--lr", "1e-4"],
    enabled=False)

## Lo que viene después

Ya tenemos: imágenes (sintéticas + fotos reales curadas), una función que las convierte en
números, y tablas de entrenamiento construidas con cuidado de no hacer trampas entre train y test.

Con esto, el siguiente cuaderno (`pycon26_02_entrenamiento.ipynb`) entrena dos modelos distintos
sobre esta misma tabla — uno clásico (Random Forest) y uno de deep learning (una CNN híbrida) — y
los evalúa de forma honesta.